# 116 — LlamaIndex Agent: ReAct over a Multi-Document Collection
## What you'll learn: QueryEngineTool, VectorStoreIndex, and the hidden ReAct loop
⏱ ~50 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/116-llamaindex-agent/llamaindex_agent_workbook.ipynb)

LlamaIndex started as a RAG-focused data framework and grew into a full agent toolkit. Its `ReActAgent` wraps the Thought/Action/Observation loop inside a single `agent.chat()` call — giving you a powerful multi-document retrieval agent with minimal boilerplate.

This workshop builds 3 `VectorStoreIndex` instances (science, history, technology), wraps each as a `QueryEngineTool`, and runs multi-hop questions that require routing across multiple tools.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — LlamaIndex primitives vs LangChain RAG |
| 2 | **Setup** — install, API key |
| 3 | **Document indexing** — VectorStoreIndex from Document objects |
| 4 | **QueryEngineTool** — wrapping indexes as agent tools |
| 5 | **ReActAgent** — the hidden ReAct loop |
| 6 | **Multi-hop questions** — routing across 3 tools |
| 7 | **LlamaIndex vs LangGraph** — what's hidden vs explicit |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `llama-index`, `llama-index-llms-openai`

### Key References
> [LlamaIndex docs](https://docs.llamaindex.ai/)
>
> [ReActAgent reference](https://docs.llamaindex.ai/en/stable/examples/agent/react_agent/)
>
> [QueryEngineTool docs](https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/tools/)

## Part 1 — Concepts: LlamaIndex vs LangChain RAG

### The same problem, different abstractions

Both LlamaIndex and LangChain solve retrieval-augmented generation, but they started from different assumptions:

```
LangChain RAG model:
  DocumentLoader -> TextSplitter -> Embeddings -> VectorStore -> Retriever -> Chain

  You compose each step explicitly.
  Flexibility is high; boilerplate is significant.

LlamaIndex RAG model:
  Documents -> VectorStoreIndex -> QueryEngine

  Index handles splitting, embedding, and storage internally.
  Less code; harder to customize individual steps.
```

### The ReAct loop: explicit vs hidden

The ReAct (Reason + Act) loop is the backbone of any tool-using agent:

```
Thought: I need to find information about X.
Action:  search_tool(query="X")
Observation: [search results]
Thought: Now I have the information. I can answer.
Action:  final_answer("The answer is...")
```

**LangGraph**: you define the loop as an explicit graph with conditional edges.
**LlamaIndex ReActAgent**: the loop is hidden inside `agent.chat()`. You see the final answer, not the intermediate steps (unless `verbose=True`).

### LlamaIndex primitives

| Primitive | Purpose |
|-----------|---------|
| `Document` | A chunk of text with optional metadata |
| `VectorStoreIndex` | Indexes Documents with embeddings for semantic search |
| `QueryEngine` | Runs retrieval + generation over an index |
| `QueryEngineTool` | Wraps a QueryEngine as an agent-callable tool |
| `ReActAgent` | Runs the ReAct loop over a list of tools |

### When to use LlamaIndex

- You have heterogeneous document collections (PDFs, web pages, databases)
- You want retrieval-first with minimal boilerplate
- You don't need fine-grained control over the ReAct loop
- You're building a document Q&A or knowledge base agent

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "llama-index==0.12.4",
         "llama-index-llms-openai==0.4.7",
         "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install (using requirements.txt)")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 3 — Document Indexing with VectorStoreIndex

### What VectorStoreIndex does internally

When you call `VectorStoreIndex.from_documents(docs)`:

```
for each Document:
  1. Split into chunks (default: 1024 tokens with 20 overlap)
  2. Embed each chunk using text-embedding-ada-002
  3. Store (chunk, embedding) in an in-memory vector store

At query time:
  1. Embed the query
  2. Find top-k most similar chunks (cosine similarity)
  3. Pass chunks + query to LLM for answer synthesis
```

You don't see any of this — it's all in `from_documents()`.

### In-memory vs persistent

By default, LlamaIndex stores embeddings in RAM. For production, you'd pass a storage context pointing to a vector database (Chroma, Pinecone, Weaviate, etc.):

```python
from llama_index.core import StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(docs, storage_context=storage_context)
```

For this workshop we use the in-memory default.

In [ ]:
from llama_index.core import Document, VectorStoreIndex

# Science documents
SCIENCE_DOCS = [
    Document(text=(
        "Photosynthesis is the process by which plants convert sunlight, water, and "
        "CO2 into glucose and oxygen. It occurs in chloroplasts using chlorophyll. "
        "The light-dependent reactions occur in the thylakoid membrane; the Calvin "
        "cycle occurs in the stroma. Plants produce ~120 billion tonnes of biomass/year."
    )),
    Document(text=(
        "DNA (deoxyribonucleic acid) stores genetic information as sequences of four "
        "bases: adenine, thymine, guanine, and cytosine. The human genome contains "
        "~3 billion base pairs encoding ~20,000 protein-coding genes. Replication is "
        "semiconservative: each old strand templates a new complementary strand."
    )),
    Document(text=(
        "Quantum mechanics describes matter and energy at the atomic scale. Key principles: "
        "wave-particle duality, superposition (Schrodinger, 1926), and the uncertainty "
        "principle (Heisenberg, 1927). Quantum entanglement links particle states regardless "
        "of distance. Applications: transistors, lasers, MRI, quantum computing."
    )),
]

# Index science documents
print("Indexing science documents...")
science_index = VectorStoreIndex.from_documents(SCIENCE_DOCS)
print(f"Science index ready. Node count: {len(science_index.docstore.docs)}")

# Quick test: query the index directly (without agent)
science_engine = science_index.as_query_engine()
test_q = "What is the uncertainty principle and who formulated it?"
resp = science_engine.query(test_q)
print(f"\nTest query: {test_q}")
print(f"Answer: {str(resp)[:300]}")

In [ ]:
from llama_index.core import VectorStoreIndex, Document

# History documents
HISTORY_DOCS = [
    Document(text=(
        "The Roman Empire at its peak (117 AD under Trajan) controlled 5 million km2 "
        "and 70 million people — 21% of the world's population. Rome transitioned from "
        "republic to empire in 27 BC when Octavian became Augustus. The Western Roman "
        "Empire fell in 476 AD; the Byzantine Empire survived until 1453."
    )),
    Document(text=(
        "The Industrial Revolution began in Britain around 1760. Key innovations: "
        "the steam engine (Watt, 1769), spinning jenny (Hargreaves, 1764), iron smelting "
        "with coke (Darby, 1709). Coal output rose 8x between 1700 and 1800. "
        "By 1850, half of Britain's population lived in cities."
    )),
    Document(text=(
        "World War II (1939-1945) was the deadliest conflict in human history: 70-85M "
        "fatalities. Germany invaded Poland on September 1, 1939. The Allies defeated the "
        "Axis powers. V-E Day: May 8, 1945. V-J Day: September 2, 1945, after atomic "
        "bombs on Hiroshima (Aug 6) and Nagasaki (Aug 9)."
    )),
]

TECHNOLOGY_DOCS = [
    Document(text=(
        "The Internet originated from ARPANET (1969). Tim Berners-Lee invented the "
        "World Wide Web in 1989. TCP/IP is the underlying protocol suite. As of 2024, "
        "5.4 billion people (67% of global population) use the internet."
    )),
    Document(text=(
        "Machine learning (ML) is a subset of AI where systems learn from data. "
        "Three paradigms: supervised learning, unsupervised learning, reinforcement "
        "learning. Deep learning uses neural networks with many layers. Transformers "
        "(2017) power modern LLMs like GPT-4 and Claude."
    )),
    Document(text=(
        "Blockchain is a distributed ledger where records are linked via cryptographic "
        "hashes. Bitcoin (Nakamoto, 2008) was the first blockchain application. "
        "Ethereum (2015) introduced smart contracts. Proof-of-stake reduces energy "
        "consumption by 99.9% vs proof-of-work."
    )),
]

print("Indexing history documents...")
history_index = VectorStoreIndex.from_documents(HISTORY_DOCS)
print(f"History index ready. Nodes: {len(history_index.docstore.docs)}")

print("\nIndexing technology documents...")
tech_index = VectorStoreIndex.from_documents(TECHNOLOGY_DOCS)
print(f"Technology index ready. Nodes: {len(tech_index.docstore.docs)}")

## Part 4 — QueryEngineTool: Wrapping Indexes as Agent Tools

### What QueryEngineTool adds

A raw `QueryEngine` is just a function: `query(question) -> response`. The agent doesn't know what it's good for.

`QueryEngineTool` adds a `ToolMetadata` — a name and description that the agent uses to decide whether to call this tool for a given question:

```python
QueryEngineTool(
    query_engine=science_index.as_query_engine(),
    metadata=ToolMetadata(
        name="science_tool",
        description="Answers questions about biology, chemistry, and physics.",
    ),
)
```

The agent reads the description and decides: "This question is about photosynthesis — that's science, so I should use `science_tool`."

**Quality tip**: The description is the most important field. A bad description means the agent routes to the wrong tool.

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

science_tool = QueryEngineTool(
    query_engine=science_index.as_query_engine(),
    metadata=ToolMetadata(
        name="science_tool",
        description=(
            "Answers questions about scientific topics including biology (photosynthesis, DNA), "
            "physics (quantum mechanics, uncertainty principle), and chemistry."
        ),
    ),
)

history_tool = QueryEngineTool(
    query_engine=history_index.as_query_engine(),
    metadata=ToolMetadata(
        name="history_tool",
        description=(
            "Answers questions about historical events including the Roman Empire, "
            "the Industrial Revolution, and World War II."
        ),
    ),
)

tech_tool = QueryEngineTool(
    query_engine=tech_index.as_query_engine(),
    metadata=ToolMetadata(
        name="technology_tool",
        description=(
            "Answers questions about technology including the Internet, machine learning, "
            "deep learning, transformers, blockchain, and AI."
        ),
    ),
)

tools = [science_tool, history_tool, tech_tool]
print(f"Created {len(tools)} QueryEngineTool instances:")
for t in tools:
    print(f"  {t.metadata.name}: {t.metadata.description[:60]}...")

## Part 5 — ReActAgent: The Hidden ReAct Loop

### How ReActAgent.from_tools() works

```python
agent = ReActAgent.from_tools(
    tools=[science_tool, history_tool, tech_tool],
    llm=OpenAI(model="gpt-4o-mini"),
    verbose=True,   # shows Thought/Action/Observation steps
    max_iterations=8,
)
```

When you call `agent.chat("question")`:

```
Iteration 1:
  Thought: "This is about quantum mechanics — I should use science_tool."
  Action:  science_tool.query("what is the uncertainty principle?")
  Observation: [retrieved text from science index]

Iteration 2:
  Thought: "I also need history context — when was it formulated?"
  Action:  science_tool.query("Heisenberg uncertainty principle year")
  Observation: [more retrieved text]

Iteration 3:
  Thought: "I have enough information to answer."
  Response: [final synthesized answer]
```

### max_iterations guard

Without `max_iterations`, a poorly calibrated model might loop indefinitely.
Setting `max_iterations=8` ensures the agent terminates even if it can't find
a satisfying answer.

In [ ]:
from llama_index.core.agent import ReActAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini", temperature=0)

agent = ReActAgent.from_tools(
    tools=tools,
    llm=llm,
    verbose=False,  # set True to see Thought/Action/Observation steps
    max_iterations=8,
)

# Single-hop test: pure science question
q1 = "How does photosynthesis produce oxygen?"
print(f"Q (science only): {q1}")
r1 = agent.chat(q1)
print(f"A: {str(r1)[:400]}")

## Part 6 — Multi-Hop Questions Across Tools

Multi-hop questions require the agent to gather information from multiple tools before synthesizing an answer. This is where `QueryEngineTool` routing really shines.

### Question design

Each question below is designed to require at least 2 tools:

1. **Science + History**: "What physical principle makes quantum computing possible, and when was it first described?" → needs science (quantum mechanics) + science again (Heisenberg 1927)

2. **History + Science**: "The Industrial Revolution burned coal. What biological process captured the energy in that coal originally?" → needs history (Industrial Revolution) + science (photosynthesis)

3. **Technology synthesis**: "Bitcoin was invented in 2008. What percentage of people use the Internet today, and what technology underlies both?" → needs technology tool for both sub-questions

In [ ]:
MULTI_HOP_QUESTIONS = [
    (
        "What physical principle makes quantum computing fundamentally different from "
        "classical computing, and when was that principle first described?",
        "Routes: science_tool (quantum) + science_tool (Heisenberg date)"
    ),
    (
        "The Industrial Revolution depended on coal. What biological process originally "
        "captured the energy stored in that coal, and how does it work?",
        "Routes: history_tool (Industrial Revolution) + science_tool (photosynthesis)"
    ),
    (
        "Bitcoin was introduced in 2008. What percentage of the global population "
        "uses the Internet today, and what do blockchain and the Internet have in common?",
        "Routes: technology_tool (blockchain) + technology_tool (internet stats)"
    ),
]

results = []
for question, routing_note in MULTI_HOP_QUESTIONS:
    print(f"\n{'='*65}")
    print(f"Q: {question}")
    print(f"Expected routing: {routing_note}")
    print("-"*65)
    response = agent.chat(question)
    answer = str(response)
    results.append({"question": question, "answer": answer})
    print(f"A: {answer[:500]}")
    if len(answer) > 500:
        print("... [truncated]")

## Part 7 — LlamaIndex vs LangGraph: What's Hidden vs Explicit

### The visibility tradeoff

```
LlamaIndex ReActAgent:

  agent = ReActAgent.from_tools([...])
  response = agent.chat("complex multi-hop question")
  # Done. 2 lines. The ReAct loop, tool routing, and synthesis are hidden.

  Pros: minimal boilerplate, fast to build
  Cons: hard to intercept, customize, or debug individual steps

LangGraph equivalent:

  graph = StateGraph(AgentState)
  graph.add_node("reason", reasoning_node)       # Thought step
  graph.add_node("call_tool", tool_execution)    # Action step
  graph.add_node("observe", observation_node)    # Observation step
  graph.add_conditional_edges("observe", route)  # Loop or END
  app = graph.compile()
  result = app.invoke({"question": "..."})

  Pros: full control, checkpointing, streaming, interrupt-resume
  Cons: 20-40 lines for equivalent functionality
```

### Decision framework

Use **LlamaIndex ReActAgent** when:
- Speed of development matters more than fine-grained control
- You're primarily doing document Q&A over heterogeneous sources
- You don't need custom loop logic (e.g., memory, retry, human-in-the-loop)

Use **LangGraph** when:
- You need deterministic, auditable execution
- Multi-agent systems with explicit handoffs
- Streaming partial results to a UI
- Checkpointing and resuming long-running tasks

In [ ]:
# Side-by-side code comparison

LLAMAINDEX_CODE = '''
# LlamaIndex (hidden loop)
from llama_index.core.agent import ReActAgent
from llama_index.llms.openai import OpenAI

agent = ReActAgent.from_tools(
    tools=[science_tool, history_tool, tech_tool],
    llm=OpenAI(model="gpt-4o-mini"),
)
response = agent.chat("Which came first: photosynthesis or coal?")
# Done — loop is invisible
'''

LANGGRAPH_CODE = '''
# LangGraph (explicit loop)
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class AgentState(TypedDict):
    question: str
    thoughts: list[str]
    tool_results: list[str]
    answer: str
    iterations: int

graph = StateGraph(AgentState)
graph.add_node("think", think_node)       # Thought step
graph.add_node("act", act_node)           # Action step
graph.add_node("observe", observe_node)   # Observation step
graph.add_conditional_edges("observe", route_fn)
app = graph.compile()
result = app.invoke({"question": "...", "thoughts": [], ...})
# Every step is visible, interceptable, checkpointable
'''

print("LlamaIndex ReActAgent (~8 lines):")
print(LLAMAINDEX_CODE)
print("\nLangGraph equivalent (~25 lines):")
print(LANGGRAPH_CODE)

## Exercises

### Exercise 1 — Add a fourth tool

Create a new `Document` about climate science (include facts about CO2, global warming, IPCC).
Index it with `VectorStoreIndex.from_documents([climate_doc])`, wrap as a `QueryEngineTool`,
add it to the agent, and ask: "How does the burning of coal during the Industrial Revolution
relate to climate change today?"

---

### Exercise 2 — verbose=True inspection

Re-run one of the multi-hop questions with `verbose=True` to see the full ReAct loop.
How many Thought/Action/Observation cycles does the agent use? Does it always call exactly
the tools you'd expect?

---

### Exercise 3 — Tool description quality

Take the `technology_tool` and change its description to something vague:
`"Answers some questions about some topics."` Re-run the Bitcoin question. Does the agent
still route correctly? What does this tell you about description quality?

In [ ]:
# ===== ANSWER KEY — Exercise 1: Fourth climate tool =====

climate_doc = Document(text=(
    "Climate change is driven primarily by burning fossil fuels (coal, oil, gas), "
    "which releases CO2 into the atmosphere. Since pre-industrial times, CO2 has "
    "risen from 280 ppm to 421 ppm (2024). The IPCC reports that global average "
    "temperature has risen 1.1C. Limiting warming to 1.5C requires net-zero emissions "
    "by 2050. Coal combustion is the single largest source of CO2 emissions globally."
))

climate_index = VectorStoreIndex.from_documents([climate_doc])
climate_tool = QueryEngineTool(
    query_engine=climate_index.as_query_engine(),
    metadata=ToolMetadata(
        name="climate_tool",
        description=(
            "Answers questions about climate change, CO2 emissions, global warming, "
            "fossil fuels, and IPCC findings."
        ),
    ),
)

# Create new agent with 4 tools
agent_v2 = ReActAgent.from_tools(
    tools=[science_tool, history_tool, tech_tool, climate_tool],
    llm=llm,
    verbose=False,
    max_iterations=10,
)

cross_domain_q = (
    "How does the burning of coal during the Industrial Revolution "
    "relate to climate change today?"
)
print(f"Q: {cross_domain_q}")
resp = agent_v2.chat(cross_domain_q)
print(f"\nA: {str(resp)[:600]}")

In [ ]:
# ===== ANSWER KEY — Exercise 2: verbose=True inspection =====

verbose_agent = ReActAgent.from_tools(
    tools=tools,
    llm=llm,
    verbose=True,  # shows Thought/Action/Observation steps
    max_iterations=8,
)

print("Running with verbose=True to see the ReAct loop steps:\n")
print("="*65)
q = "What physical principle makes quantum computing special?"
resp = verbose_agent.chat(q)
print("="*65)
print(f"\nFinal answer: {str(resp)[:400]}")
print("\nNote: count the Thought/Action/Observation cycles above.")

In [ ]:
# ===== ANSWER KEY — Exercise 3: Tool description quality =====

# Deliberately vague description
vague_tech_tool = QueryEngineTool(
    query_engine=tech_index.as_query_engine(),
    metadata=ToolMetadata(
        name="technology_tool",
        description="Answers some questions about some topics.",  # deliberately bad
    ),
)

agent_vague = ReActAgent.from_tools(
    tools=[science_tool, history_tool, vague_tech_tool],
    llm=llm,
    verbose=False,
    max_iterations=8,
)

q_bitcoin = "Bitcoin was introduced in 2008. What percentage of people use the Internet today?"
print(f"Q: {q_bitcoin}")
print("\nWith VAGUE description:")
resp_vague = agent_vague.chat(q_bitcoin)
print(f"A: {str(resp_vague)[:400]}")

print("\nWith GOOD description:")
resp_good = agent.chat(q_bitcoin)
print(f"A: {str(resp_good)[:400]}")
print()
print("Lesson: vague descriptions cause misrouting or failure to use the right tool.")
print("The description is the agent's only signal for tool selection.")

## Workshop Complete

You have built a full LlamaIndex multi-document agent:

- **`Document`** — text units with optional metadata
- **`VectorStoreIndex`** — handles splitting, embedding, and storage automatically
- **`QueryEngineTool`** — adds name + description so the agent can route queries
- **`ReActAgent`** — executes the hidden Thought/Action/Observation loop
- **Tool description quality** — the most impactful variable in routing accuracy

**Key insight**: LlamaIndex hides the ReAct loop behind `agent.chat()`. This reduces boilerplate but removes visibility and control. LangGraph makes the same loop explicit as nodes and edges — more code, but full control over every step, with checkpointing and streaming built in.

---

Next: **example 117** — Instructor: type-safe structured extraction with automatic retry on validation failure.

---
### Further reading
- [LlamaIndex ReActAgent](https://docs.llamaindex.ai/en/stable/examples/agent/react_agent/)
- [QueryEngineTool](https://docs.llamaindex.ai/en/stable/module_guides/deploying/agents/tools/)
- [VectorStoreIndex](https://docs.llamaindex.ai/en/stable/module_guides/indexing/vector_store_index/)
- [LlamaIndex vs LangChain comparison](https://docs.llamaindex.ai/en/stable/getting_started/concepts/)</cell id="cell-md-021"></cell>
